# Third Party Data (3PD) Example Notebook

Send audience segments you own as a data provider to The Trade Desk for monetization.

## Is this the right notebook for you?

| | |
|---|---|
| **Who Should Use This?** | Third-party data providers and commerce partners |
| **What Does the Notebook Do?** | Adds identities to a third-party audience segment available in the TTD marketplace |
| **What Data Does It Send?** | One row per identity per segment |
| **Destination Trade Desk Endpoint** | `POST /data/thirdparty` (via the `ThirdPartyContext` class of the ttd-databricks SDK) |
| **Relevant OpenTTD API Documentation** | [Third-party data](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/audience/doc/post-data-thirdparty) |

Uploading data on behalf of a single advertiser instead? That is first-party data, so use
[First Party Data (1PD) Example Notebook](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/example_notebook/First%20Party%20Data%20%281PD%29%20Example%20Notebook.ipynb).

## What you need

- [ ] A TTD Platform API token — see [Create an API token](https://open.thetradedesk.com/advertiser/docsApp/GuidesAdvertiser/data/doc/DataApiCallsAdvertiser#ui-method-create)
- [ ] Your data provider ID
- [ ] (Optional) A UID2 operator base URL, API key and client secret — to send raw email addresses or phone numbers

## Step 1: Install the SDK

In [ ]:
%pip install ttd-databricks

dbutils.library.restartPython()

## Step 2: Configure credentials

For a first run you can paste values inline. In production, read them from Databricks Secrets:

```python
API_TOKEN        = dbutils.secrets.get(scope="ttd", key="api-token")
DATA_PROVIDER_ID = dbutils.secrets.get(scope="ttd", key="data-provider-id")
```

> **Note:** Authenticate with a Platform API token, sent as the `TTD-Auth` header. Generate one
> in the OpenTTD Access Management app — see [Create an API token](https://open.thetradedesk.com/advertiser/docsApp/GuidesAdvertiser/data/doc/DataApiCallsAdvertiser#ui-method-create). Secret keys and
> `TtdSignature` headers are a legacy method and are not supported by this SDK.

In [ ]:
API_TOKEN        = "<your-ttd-auth-token>"
DATA_PROVIDER_ID = "<your-data-provider-id>"

## Step 3: Create the client and context

In [ ]:
from ttd_databricks_python.ttd_databricks import (
    ThirdPartyContext,
    TTDEndpoint,
    TtdDatabricksClient,
    get_ttd_input_schema,
)

# Two ways to create the client — this notebook uses (ii) everywhere below.
# i)  Dependency injection — you build the DataClient and pass it in:
#         from ttd_data import DataClient
#         client = TtdDatabricksClient(data_api_client=DataClient(), api_token=API_TOKEN)
# ii) Factory — from_params builds the DataClient for you:
client = TtdDatabricksClient.from_params(api_token=API_TOKEN)

context = ThirdPartyContext(
    data_provider_id=DATA_PROVIDER_ID,
    # Set True only if id_value already holds a hashed identifier.
    is_user_id_already_hashed=False,
)

print(f"Context: {context}")

## Step 4: Inspect the required input schema

Mandatory columns: `id_type`, `id_value`, `segment_name`

Optional columns: `cookie_mapping_partner_id`, `timestamp_utc`, `ttl_in_minutes`

In [ ]:
from ttd_databricks_python.ttd_databricks import TTDEndpoint, get_ttd_input_schema
from ttd_databricks_python.ttd_databricks.schemas import get_required_column_names

input_schema = get_ttd_input_schema(TTDEndpoint.THIRD_PARTY)

print("Mandatory columns:", get_required_column_names(TTDEndpoint.THIRD_PARTY))
print("\nFull input schema:")
for field in input_schema.fields:
    print(f"  {field.name}: {field.dataType.simpleString()} (nullable={field.nullable})")

## Step 5: Prepare your input DataFrame

One row per identity per segment. `segment_name` is the data element the identity joins.

> **Tip:** Start with a handful of rows. Confirming the end-to-end flow on a small sample is much easier to troubleshoot than a full load.

In [ ]:
rows = [
    {"id_type": "TDID",    "id_value": "123e4567-e89b-12d3-a456-426652340000",
     "segment_name": "1210", "ttl_in_minutes": 43200},
    {"id_type": "DAID",    "id_value": "a9342d1f-69f1-4bf8-bc2b-1f20eb451f21",
     "segment_name": "1150", "ttl_in_minutes": 43200},
    {"id_type": "UID2",    "id_value": "48MjlfIUZpOKNAm9nod7/jCLAXUYsnE1tpVHQSDS0uo=",
     "segment_name": "1630", "ttl_in_minutes": 43200},
    {"id_type": "ID5",     "id_value": "ID5-c62drGF0EC6wsCZVFDbTbZwi33eB0uZTIC8FxJpzsQ",
     "segment_name": "1800", "ttl_in_minutes": 43200},
    {"id_type": "FirstId", "id_value": "8934d279bba4c7d652a02f624dc334e3",
     "segment_name": "1810", "ttl_in_minutes": 43200},
]

input_df = spark.createDataFrame(rows, schema=input_schema)
display(input_df)

## Step 6: Sending Data

There are two ways to send your data. Pick one — you do not need both.

| | Ad hoc | Batch processing |
|---|---|---|
| **State management** | None. Every call sends every row you give it. | Provided. A metadata table records progress, so each run sends only rows added since the last one. |
| **Input** | A DataFrame you build in the notebook | A Delta input table |
| **Output** | Returned inline as a DataFrame | Written to a Delta output table |
| **Best for** | One-off loads and first tests | Recurring pipelines |

### Step 6a: Ad Hoc Usage (No State Management Provided)

`push_data` sends your DataFrame straight to the Data API and returns the input columns
enriched with per-row status. The SDK keeps no record of what it has already sent, so
re-running this cell sends every row again.

It does not raise on API or row-level failures — every outcome is reported inline.

In [ ]:
result_df = client.push_data(df=input_df, context=context, batch_size=1600)

display(result_df)

`push_data` adds these columns to your input:

| Column | Meaning |
|---|---|
| `success` | `True` if the row was accepted |
| `error_code` | Failure category, `null` on success |
| `error_message` | Human-readable reason, `null` on success |
| `processed_timestamp` | When the row was submitted |
| `uid2_resolutions` | Raw identifier → UID2 mapping, empty unless `uid2_config` was set |

In [ ]:
from pyspark.sql.functions import col

total     = result_df.count()
succeeded = result_df.filter(col("success")).count()

print(f"Total: {total} | Succeeded: {succeeded} | Failed: {total - succeeded}")

failed_df = result_df.filter(~col("success"))
if failed_df.count():
    display(failed_df.select("error_code", "error_message"))

### Step 6b: Batch Processing (State Management Provided)

`batch_process` reads from a Delta input table, writes results to an output table, and
records how far it got in a metadata table. With `process_new_records_only=True`, each run
picks up only the rows added since the last successful run, so you can schedule it without
re-sending history.

**One time steps:** Create the three Delta tables. Every future run reuses these same tables —
the metadata table is what remembers your progress, so do not drop or recreate it between
runs. The `setup_*` methods are safe to re-run: they return the existing table if it is
already there.

In [ ]:
from ttd_databricks_python.ttd_databricks import TTDEndpoint

input_table    = client.setup_input_table(endpoint=TTDEndpoint.THIRD_PARTY)
output_table   = client.setup_output_table(endpoint=TTDEndpoint.THIRD_PARTY)
metadata_table = client.setup_metadata_table()

print(f"Input table:    {input_table}")
print(f"Output table:   {output_table}")
print(f"Metadata table: {metadata_table}")

**Every run:**

**1. Append new rows to the input table.** In production this is your upstream pipeline's job.

In [ ]:
from pyspark.sql import functions as F

(
    spark.createDataFrame(rows, schema=input_schema)
         .withColumn("updated_at", F.current_timestamp())
         .write.format("delta").mode("append").saveAsTable(input_table)
)

display(spark.table(input_table))

**2. Call `batch_process`.** Re-running it picks up only rows appended since the last run.

In [ ]:
client.batch_process(
    context=context,
    input_table=input_table,
    output_table=output_table,
    metadata_table=metadata_table,
    process_new_records_only=True,  # incremental; set False to reprocess every row
    batch_size=1600,                # rows per API request
)

display(spark.table(output_table))
display(spark.table(metadata_table))

## (Optional) Sending email addresses or phone numbers (UID2)

Skip this section if you are sending device IDs or UID2s you have already resolved.

Pass a `uid2_config` when you create the client. It is the same call as Step 3 with one
extra argument. Then set `id_type` to `Email`, `Phone`,
`HashedEmail`, or `HashedPhone` on your rows. From there `push_data` and `batch_process` take them exactly like any other
identifier type, and nothing else about Step 6a or Step 6b changes.

The SDK resolves each identifier to a UID2 (or EUID) using your operator before the
request leaves Databricks, so The Trade Desk never receives the raw email or phone
number. The mapping comes back in the `uid2_resolutions` column.

In [ ]:
from ttd_data.uid2 import IdentityScope, UID2Config

from ttd_databricks_python.ttd_databricks import TtdDatabricksClient

uid2_client = TtdDatabricksClient.from_params(
    api_token=API_TOKEN,
    uid2_config=UID2Config(
        base_url="<your-uid2-operator-url>",
        api_key="<your-uid2-api-key>",
        client_secret="<your-uid2-client-secret>",
        identity_scope=IdentityScope.UID2,  # use IdentityScope.EUID for European identities
    ),
)

uid2_data = [
    {"id_type": "Email",       "id_value": "user@example.com",
     "segment_name": "1210", "ttl_in_minutes": 43200},
    {"id_type": "HashedEmail", "id_value": "tMmiiTI7IaAcPpQPFQ65uMVCWH8av9jw4cwf/F5HVRQ=",
     "segment_name": "1210", "ttl_in_minutes": 43200},
]

uid2_result_df = uid2_client.push_data(
    df=spark.createDataFrame(uid2_data, schema=input_schema),
    context=context,
)

display(uid2_result_df.select("id_type", "success", "error_message", "uid2_resolutions"))

## Next steps

- **Other use cases** — one notebook per use case:
  [First Party Data (1PD) Example Notebook](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/example_notebook/First%20Party%20Data%20%281PD%29%20Example%20Notebook.ipynb),
  [Third Party Data (3PD) Example Notebook](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/example_notebook/Third%20Party%20Data%20%283PD%29%20Example%20Notebook.ipynb),
  [Offline Conversion Data (CAPI) Example Notebook](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/example_notebook/Offline%20Conversion%20Data%20%28CAPI%29%20Example%20Notebook.ipynb),
  [Deletion and Opt-Out (DSR) Example Notebook](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/example_notebook/Deletion%20and%20Opt-Out%20%28DSR%29%20Example%20Notebook.ipynb).
- **Full reference** — [README](https://github.com/thetradedesk/ttd-databricks-python/blob/add-easy-start-examples-per-usecase-to-sdk-docs/README.md) covers authentication, error handling,
  UID2 support, custom HTTP clients, and server URL overrides.
- **Server URLs** — you do not need to configure one. Each endpoint already targets its
  own default server, and it can be overridden with a preferred server if you need one.